# Notebook 02: Agro-Climatic Feature Engineering & Hypothesis Testing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fahadrahman444/scientific-research-and-methodologies/blob/main/colab_notebooks/02_Feature_Engineering_and_Statistical_Analysis.ipynb)

**Research Topic:** A Data-Driven Analysis of 15-Year Weather Variability and Its Impact on Agricultural Crop Yield in Bangladesh  
**Methodological Framework:** Two-Tier Quantitative Design (Option A)  
**Course:** CSE 418: Scientific Research and Methodologies Lab (UITS)  

---
## 1. Overview
Computes Mann-Kendall non-parametric trend tests, Sen's slope rates, Variance Inflation Factor (VIF) collinearity diagnostics, and isolates climate signals from technological adoption.


In [ ]:
# Step 1: Environment & Data Ingestion
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

INTEGRATED_LOCAL = '../data/processed/bangladesh_faostat_weather_integrated_15yr.csv'
INTEGRATED_URL = f'{BASE_RAW}/data/processed/bangladesh_faostat_weather_integrated_15yr.csv'
REGIONAL_LOCAL = '../data/processed/bangladesh_weather_crop_15yr_processed.csv'
REGIONAL_URL = f'{BASE_RAW}/data/processed/bangladesh_weather_crop_15yr_processed.csv'

df_nat = pd.read_csv(INTEGRATED_LOCAL) if os.path.exists(INTEGRATED_LOCAL) else pd.read_csv(INTEGRATED_URL)
df_reg = pd.read_csv(REGIONAL_LOCAL) if os.path.exists(REGIONAL_LOCAL) else pd.read_csv(REGIONAL_URL)
print('Datasets loaded successfully.')


In [ ]:
# Step 2: Implementation of Mann-Kendall Trend Test and Sen's Slope
def mann_kendall(x, alpha=0.05):
    n = len(x)
    s = 0
    for k in range(n - 1):
        for j in range(k + 1, n):
            s += np.sign(x[j] - x[k])
    var_s = (n * (n - 1) * (2 * n + 5)) / 18.0
    z = (s - 1) / np.sqrt(var_s) if s > 0 else ((s + 1) / np.sqrt(var_s) if s < 0 else 0)
    p_val = 2 * (1 - stats.norm.cdf(abs(z)))
    h = abs(z) > stats.norm.ppf(1 - alpha / 2)
    slopes = [(x[j] - x[k]) / (j - k) for k in range(n - 1) for j in range(k + 1, n)]
    return z, p_val, np.median(slopes), 'Statistically Significant' if h else 'No Monotonic Trend'

# Tier 1: National Trends (FAOSTAT)
nat_vars = ['yield_kg_ha', 'tmax', 'tmin', 'temp_mean', 'rain', 'heat_days']
nat_records = []
for v in nat_vars:
    z, p, slope, status = mann_kendall(df_nat[v].values)
    nat_records.append({'Variable': v, 'Z_Score': round(z, 3), 'p_value': round(p, 4), 'Sen_Slope': round(slope, 4), 'Status': status})

print('=== TIER 1: MANN-KENDALL TESTS ON NATIONAL FAOSTAT & CLIMATE SERIES ===')
display(pd.DataFrame(nat_records))


In [ ]:
# Step 3: Tier 2 Divisional Trend Test (Across 8 Divisions)
div_records = []
for div in df_reg['division'].unique():
    div_df = df_reg[df_reg['division'] == div].groupby('year').mean()
    z, p, slope, status = mann_kendall(div_df['temp_mean'].values)
    div_records.append({'Division': div, 'Z_Score': round(z, 3), 'p_value': round(p, 4), 'Warming_Rate (°C/yr)': round(slope, 4), 'Status': status})

print('=== TIER 2: 15-YEAR TEMPERATURE TRENDS ACROSS 8 ADMINISTRATIVE DIVISIONS ===')
display(pd.DataFrame(div_records))


In [ ]:
# Step 4: De-trending Technological Gains (Separating HYV Breeding from Climate Shock)
x_years = df_nat['year'].values - 2009
fit = np.polyfit(x_years, df_nat['yield_kg_ha'].values, deg=1)
df_nat['secular_trend'] = np.polyval(fit, x_years)
df_nat['climate_shock_residual'] = df_nat['yield_kg_ha'] - df_nat['secular_trend']

plt.figure(figsize=(11, 5))
plt.plot(df_nat['year'], df_nat['yield_kg_ha'], 'o-', color='#2ca02c', label='Observed FAOSTAT Yield')
plt.plot(df_nat['year'], df_nat['secular_trend'], '--', color='blue', label='Technological Baseline (HYV & Irrigation)')
plt.title('Separation of Technological Agronomic Gain from Climate Shocks', fontsize=12, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Yield (kg/ha)')
plt.legend()
plt.tight_layout()
plt.show()
